# Viscan OPG — benchmark ladder: can we beat the measured Faster R-CNN result?

One Kaggle GPU session that measures the full pre-registered ladder from `RESEARCH_BENCHMARKS.md` §7:

| id | pipeline | what is new vs the deployed model |
|---|---|---|
| P0 | Faster R-CNN (downloaded, SHA-256 verified) | reference row; must reproduce the published 0.496 / 0.329 |
| P1 | + horizontal-flip TTA | 2x forward passes |
| P2 | + native-scale sliced inference (800x800 tiles, WBF with the full-image pass) | attacks the 0.474x mean downscale of DENTEX images |
| P3 | P2 + RetinaNet R50-FPN-V2 trained here, WBF over all members | architecture diversity |
| P4 | P2 + ConvNeXt-Tiny crop refiner (`rescore`) | attacks deep_caries->caries confusion and caries background FPs |
| P5 | P3 + crop refiner (`rescore`) | |
| P6 | P2 + crop refiner (`reclass`) | drops boxes the classifier calls background |
| P7 | P3 + crop refiner (`reclass`) | |

**Protocol (fixed before the run)**
- DENTEX pinned revision, `SEED=42` split: 635 train / 70 val / 50 held-out test.
- Every pipeline is fused with WBF at IoU 0.55; the operating threshold is selected on the **70-image validation split only** (macro-F1 sweep 0.10-0.90), then applied once to the test split.
- Pipeline choice uses **validation mAP@50:95 only**. All 8 test rows are reported so no result is hidden.
- The crop refiner is trained only on train-split GT crops.
- No number below is pre-filled: every metric is read from the JSON this run writes. If a stage loses, its row stays in the table.

**Before running:** Settings -> Accelerator **T4 (x1 is enough; x2 is fine)**, Settings -> Internet **On**.
RetinaNet training (12 epochs) is the long step; sliced inference adds ~10 forward passes per image.

In [ ]:
import os, sys, json, time, platform, subprocess
from pathlib import Path

import torch, torchvision

print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("platform   :", platform.platform())

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> T4, then re-run."
print("gpu        :", torch.cuda.get_device_name(0))
print("vram (GB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
REPO_URL = "https://github.com/surendra902/viscan-opg.git"
PINNED_COMMIT = "1ac108065852d42ba66bf799fc7d4b443ff49ec2"
WORKDIR = Path("/kaggle/working/viscan-opg")

if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "--quiet", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(WORKDIR)], check=True)

subprocess.run(["git", "-C", str(WORKDIR), "checkout", "--quiet", PINNED_COMMIT], check=True)
head = subprocess.run(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"],
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == PINNED_COMMIT, f"checkout mismatch: {head}"
print("repo pinned at", head)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

for mod in ("yaml", "pycocotools", "matplotlib", "pandas", "PIL"):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)

import model as viscan_model
assert viscan_model.HAS_TORCHVISION
print("imports OK | classes:", viscan_model.CLASS_NAMES)

In [ ]:
os.chdir(WORKDIR)

data_dir = Path("data/dentex")
test_images = data_dir / "images" / "test"
ready = ((data_dir / "annotations" / "instances_test.json").exists()
         and test_images.is_dir() and any(test_images.iterdir()))

if ready:
    print("[SKIP] dataset already prepared at", data_dir.resolve())
else:
    print("[RUN] downloading + converting DENTEX (pinned revision) into", data_dir.resolve())
    proc = subprocess.run(["bash", "-lc",
        f'set -o pipefail; "{sys.executable}" prepare_data.py data/dentex 2>&1 | tee prepare_data.log'],
        check=False)
    assert proc.returncode == 0, "prepare_data.py failed - check prepare_data.log"

In [ ]:
os.chdir(WORKDIR)
import pandas as pd

data_dir = Path("data/dentex")
summary = {}
for s in ("train", "val", "test"):
    coco = json.loads((data_dir / "annotations" / f"instances_{s}.json").read_text(encoding="utf-8"))
    per_class = {cid: 0 for cid in viscan_model.CLASS_NAMES if cid != 0}
    for ann in coco["annotations"]:
        per_class[ann["category_id"]] += 1
    summary[s] = {
        "images": len(coco["images"]),
        "boxes": len(coco["annotations"]),
        "per_class": per_class,
        "files": {img["file_name"] for img in coco["images"]},
        "scales": [min(800.0 / min(i["width"], i["height"]), 1333.0 / max(i["width"], i["height"]))
                   for i in coco["images"]],
    }

rows = []
for s, d in summary.items():
    row = {"split": s, "images": d["images"], "boxes": d["boxes"]}
    row.update({viscan_model.CLASS_NAMES[cid]: n for cid, n in d["per_class"].items()})
    row["detector scale mean"] = round(sum(d["scales"]) / len(d["scales"]), 3)
    rows.append(row)
display(pd.DataFrame(rows).set_index("split"))

assert summary["train"]["images"] == 635 and summary["val"]["images"] == 70
assert summary["test"]["images"] == 50
assert summary["train"]["boxes"] + summary["val"]["boxes"] == 3529
assert summary["test"]["boxes"] == 182
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (summary[a]["files"] & summary[b]["files"]), f"{a}/{b} image overlap"
print("split verified: 635/70/50 images, 3529/182 boxes, no overlap")

In [ ]:
os.chdir(WORKDIR)
import hashlib
from urllib.request import urlretrieve

WEIGHTS_URL = "https://huggingface.co/surendra08/viscan-opg-weights/resolve/main/best_model.pth"
WEIGHTS_SHA256 = "e0b49b12f48fc71ec1f583db08f107650659571e7c2b9cbd454dbfed13b46a1f"

ckpt = Path("checkpoints/best_model.pth")
ckpt.parent.mkdir(parents=True, exist_ok=True)
if not ckpt.exists():
    print("[DOWNLOAD]", WEIGHTS_URL)
    urlretrieve(WEIGHTS_URL, ckpt)

digest = hashlib.sha256()
with open(ckpt, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        digest.update(chunk)
actual = digest.hexdigest()
assert actual == WEIGHTS_SHA256, f"checkpoint hash mismatch: {actual}"
print(f"Faster R-CNN checkpoint verified: {ckpt.stat().st_size:,} bytes | sha256 {actual}")

In [ ]:
os.chdir(WORKDIR)

def run(cmd, log):
    proc = subprocess.run(["bash", "-lc", f'set -o pipefail; {cmd} 2>&1 | tee {log}'], check=False)
    assert proc.returncode == 0, f"command failed, inspect {log}"
    print("done ->", log)

# P0: deployed Faster R-CNN, no TTA. Dumps val + test predictions for fusion and re-scoring.
run(f'"{sys.executable}" evaluate.py --weights checkpoints/best_model.pth '
    f'--config configs/baseline.yaml --data_dir data/dentex --output_dir evaluation/P0 '
    f'--skip_variant --select_threshold val --dump_predictions preds/frcnn', "eval_p0.log")

assert Path("preds/frcnn/predictions_val.json").exists()
assert Path("preds/frcnn/predictions_test.json").exists()
print(json.loads(Path("evaluation/P0/final_evaluation_report.json").read_text())["coco_map"]["overall"])

In [ ]:
os.chdir(WORKDIR)

# P1 ablation: horizontal-flip TTA on the full image.
run(f'"{sys.executable}" evaluate.py --weights checkpoints/best_model.pth '
    f'--config configs/baseline.yaml --data_dir data/dentex --output_dir evaluation/P1 '
    f'--skip_variant --select_threshold val --tta hflip --dump_predictions preds/frcnn_tta',
    "eval_p1.log")

In [ ]:
os.chdir(WORKDIR)

# P2: full image (hflip TTA) + native-scale 800x800 tiles, one dump per view group.
for split in ("val", "test"):
    run(f'"{sys.executable}" slice_inference.py --weights checkpoints/best_model.pth '
        f'--config configs/baseline.yaml --split {split} --data_dir data/dentex '
        f'--output_dir preds/slice --views both --tile 800 --overlap 0.25 '
        f'--tta hflip --tile_tta none --max_dets_tile 100', f"slice_{split}.log")
    meta = json.loads(Path(f"preds/slice/slice_meta_{split}.json").read_text())
    print(split, {k: meta[k] for k in ("num_images", "num_tiles", "tiles_per_image_mean",
                                      "detections", "ms_per_image")})

In [ ]:
os.chdir(WORKDIR)

# RetinaNet on the identical split / seed / schedule / input scale. Longest step (~12 epochs).
t0 = time.time()
run(f'"{sys.executable}" train.py --config configs/retinanet.yaml', "train_retinanet.log")
print(f"retinanet training wall time: {(time.time() - t0) / 60.0:.1f} min")

ckpt = Path("checkpoints/retinanet/best_model.pth")
assert ckpt.exists(), "best RetinaNet checkpoint missing"
state = torch.load(ckpt, map_location="cpu", weights_only=False)
print("best epoch:", state["epoch"], "| best val mAP@50:", round(float(state["val_map50"]), 4))
del state
torch.cuda.empty_cache()

In [ ]:
os.chdir(WORKDIR)

run(f'"{sys.executable}" evaluate.py --weights checkpoints/retinanet/best_model.pth '
    f'--config configs/retinanet.yaml --data_dir data/dentex --output_dir evaluation/retina '
    f'--skip_variant --select_threshold val --tta hflip --dump_predictions preds/retinanet_tta',
    "eval_retinanet.log")

In [ ]:
os.chdir(WORKDIR)

# Crop refiner: crop bank from train-split GT only, then 4 epochs of ConvNeXt-Tiny.
run(f'"{sys.executable}" cascade_refine.py build --data_dir data/dentex '
    f'--output_dir checkpoints/cascade', "cascade_build.log")
meta = json.loads(Path("checkpoints/cascade/crop_bank_meta.json").read_text())
assert meta["source_split"] == "train", "crop bank must come from the train split only"
print("crop bank:", meta["num_samples"], "crops |", meta["counts"], "| context", meta["context_range"])

run(f'"{sys.executable}" cascade_refine.py train --bank_dir checkpoints/cascade '
    f'--output checkpoints/cascade/convnext_cascade.pth --epochs 4 --batch_size 64', "cascade_train.log")
assert Path("checkpoints/cascade/convnext_cascade.pth").exists()

In [ ]:
os.chdir(WORKDIR)

# Apply the refiner to each member dump. Separate output dirs keep file names unique.
SOURCES = [
    ("preds/slice", "full", "preds/cascade_slice_full"),
    ("preds/slice", "tiles", "preds/cascade_slice_tiles"),
    ("preds/retinanet_tta", None, "preds/cascade_retinanet"),
]

for mode in ("rescore", "reclass"):
    for src_dir, view, out_dir in SOURCES:
        for split in ("val", "test"):
            src = (f"{src_dir}/predictions_{split}_{view}.json" if view
                   else f"{src_dir}/predictions_{split}.json")
            run(f'"{sys.executable}" cascade_refine.py apply '
                f'--checkpoint checkpoints/cascade/convnext_cascade.pth --preds {src} '
                f'--split {split} --data_dir data/dentex --output_dir {out_dir} '
                f'--mode {mode} --max_dets 100', f"cascade_{mode}_{src_dir.split('/')[-1]}_{split}.log")

print("cascade dumps:")
for p in sorted(Path("preds").glob("cascade_*/cascade_meta_*.json")):
    m = json.loads(p.read_text())
    print(f"  {p.name}: {m['num_considered']}/{m['num_input']} -> {m['num_output']} "
          f"(dropped bg={m['dropped_background']}, relabelled={m['relabelled']})")

In [ ]:
os.chdir(WORKDIR)
import io, contextlib
import pandas as pd
from pycocotools.coco import COCO

from ensemble_wbf import run_ensemble, fuse_predictions
from evaluate import run_coco_map

TEST_GT = "data/dentex/annotations/instances_test.json"
VAL_GT = "data/dentex/annotations/instances_val.json"
TEST_IMAGES = "data/dentex/images/test"
coco_val = COCO(VAL_GT)

def _silent(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

M = {
    "frcnn":               {"val": ["preds/frcnn/predictions_val.json"], "test": ["preds/frcnn/predictions_test.json"]},
    "frcnn_tta":           {"val": ["preds/frcnn_tta/predictions_val.json"], "test": ["preds/frcnn_tta/predictions_test.json"]},
    "slice":               {"val": ["preds/slice/predictions_val_full.json", "preds/slice/predictions_val_tiles.json"],
                            "test": ["preds/slice/predictions_test_full.json", "preds/slice/predictions_test_tiles.json"]},
    "retina":              {"val": ["preds/retinanet_tta/predictions_val.json"], "test": ["preds/retinanet_tta/predictions_test.json"]},
}
for view in ("full", "tiles"):
    for mode in ("rescore", "reclass"):
        key = f"casc_slice_{view}_{mode}"
        M[key] = {"val": [f"preds/cascade_slice_{view}/predictions_val_{mode}.json"],
                  "test": [f"preds/cascade_slice_{view}/predictions_test_{mode}.json"]}
for mode in ("rescore", "reclass"):
    M[f"casc_retina_{mode}"] = {"val": [f"preds/cascade_retinanet/predictions_val_{mode}.json"],
                                 "test": [f"preds/cascade_retinanet/predictions_test_{mode}.json"]}

for name, d in M.items():
    for split, paths in d.items():
        for p in paths:
            assert Path(p).exists(), f"missing dump {p}"

PIPELINES = [
    ("P0 Faster R-CNN",                    ["frcnn"], "p0"),
    ("P1 FRCNN + hflip TTA",               ["frcnn_tta"], "p1"),
    ("P2 FRCNN sliced (full+tiles)",       ["slice"], "p2"),
    ("P3 P2 + RetinaNet (WBF)",            ["slice", "retina"], "p3"),
    ("P4 P2 + cascade rescore",            ["casc_slice_full_rescore", "casc_slice_tiles_rescore"], "p4"),
    ("P5 P3 + cascade rescore",            ["casc_slice_full_rescore", "casc_slice_tiles_rescore",
                                            "casc_retina_rescore"], "p5"),
    ("P6 P2 + cascade reclass",            ["casc_slice_full_reclass", "casc_slice_tiles_reclass"], "p6"),
    ("P7 P3 + cascade reclass",            ["casc_slice_full_reclass", "casc_slice_tiles_reclass",
                                            "casc_retina_reclass"], "p7"),
]

def val_metrics(paths):
    fused, _ = fuse_predictions([json.loads(Path(p).read_text()) for p in paths])
    return _silent(run_coco_map, coco_val, fused)["overall"]

rows, reports = [], {}
for label, members, tag in PIPELINES:
    val_paths = [p for m in members for p in M[m]["val"]]
    test_paths = [p for m in members for p in M[m]["test"]]
    out_dir = f"evaluation/pipelines/{tag}"

    _silent(run_ensemble, pred_paths=test_paths, gt_path=TEST_GT, images_dir=TEST_IMAGES,
            val_pred_paths=val_paths, val_gt_path=VAL_GT, output_dir=out_dir, iou_thresh=0.55)

    rep = json.loads((Path(out_dir) / "final_evaluation_report.json").read_text())
    vm = val_metrics(val_paths)
    reports[label] = rep
    rows.append({
        "pipeline": label,
        "val AP": vm["map_50_95"], "val AP50": vm["map_50"],
        "test AP": rep["coco_map"]["overall"]["map_50_95"],
        "test AP50": rep["coco_map"]["overall"]["map_50"],
        "test AP75": rep["coco_map"]["overall"]["map_75"],
        "test AR@100": rep["coco_map"]["overall"].get("ar_100"),
        "tau": rep["threshold_selection"]["best_threshold"],
        "test macro-F1": rep["threshold_metrics"]["macro_f1"],
    })
    print(f"{label:38s} val AP={vm['map_50_95']:.4f}  val AP50={vm['map_50']:.4f}")

table = pd.DataFrame(rows).set_index("pipeline").round(4)
display(table)

# Selection uses validation mAP@50:95 only.
selected = table["val AP"].idxmax()
print(f"selected on validation AP: {selected}")

In [ ]:
os.chdir(WORKDIR)
import pandas as pd

# Reproduction check for P0 against the published report (soft: prints drift, does not abort).
PUBLISHED = {"AP50": 0.496, "AP": 0.329, "AP75": 0.383, "AR@100": 0.683}
p0 = table.loc["P0 Faster R-CNN"]
measured = {"AP50": p0["test AP50"], "AP": p0["test AP"], "AP75": p0["test AP75"], "AR@100": p0["test AR@100"]}
for k, v in measured.items():
    drift = float(v) - PUBLISHED[k]
    print(f"P0 {k:7s} measured={float(v):.4f}  published={PUBLISHED[k]:.3f}  drift={drift:+.4f}")

best = table.loc[selected]
base = table.loc["P0 Faster R-CNN"]
print("\nSelected pipeline:", selected)
print(f"  test AP50 {best['test AP50']:.4f} vs P0 {base['test AP50']:.4f} "
      f"({best['test AP50'] - base['test AP50']:+.4f})")
print(f"  test AP    {best['test AP']:.4f} vs P0 {base['test AP']:.4f} "
      f"({best['test AP'] - base['test AP']:+.4f})")
print(f"  vs published 0.496 / 0.329: AP50 {best['test AP50'] - 0.496:+.4f}, AP {best['test AP'] - 0.329:+.4f}")
print("  beats published AP50:", "YES" if best['test AP50'] > 0.496 else "NO",
      "| beats published AP:", "YES" if best['test AP'] > 0.329 else "NO")

In [ ]:
os.chdir(WORKDIR)
import pandas as pd
import numpy as np

rep = reports[selected]
per_class = pd.DataFrame(rep["coco_map"]["per_class"]).T.round(4)
display(per_class)

cm = pd.DataFrame(np.asarray(rep["threshold_metrics"]["confusion_matrix"]),
                  index=["background"] + [viscan_model.CLASS_NAMES[i] for i in range(1, 5)],
                  columns=["background"] + [viscan_model.CLASS_NAMES[i] for i in range(1, 5)])
print(f"confusion matrix at tau={rep['threshold_selection']['best_threshold']:.2f} (rows=truth, cols=prediction):")
display(cm)

# External reference: DENTEX 2023 diagnosis leaderboard (250 hidden test images, percent as published).
EXTERNAL = [
    ("Hamamci I. (HierarchicalDet)", 0.691, 37.6, 60.2, 44.0),
    ("Mei S. (YOLOrtho)", 0.5972, 37.44, 56.72, 45.49),
    ("He L. (DINO + U-Net + YOLOv8 + WBF)", 0.5921, 37.06, 54.31, 44.77),
    ("Chen S. (DiffusionDet)", 0.644, 37.0, 58.1, 42.6),
    ("Choi K. (DETDet)", 0.7585, 36.48, 57.30, 42.84),
    ("Faster R-CNN reference", 0.533, 33.2, 54.3, 38.0),
    ("RetinaNet reference", 0.587, 32.5, 54.2, 35.6),
]
display(pd.DataFrame(EXTERNAL, columns=["entry", "AR@100", "AP", "AP50", "AP75"]).set_index("entry"))
print("Context only: different test images (250 hidden vs our 50 public), 705 vs 635 training images.")
print("Our rows in the same units:")
display(table[["test AR@100", "test AP", "test AP50", "test AP75"]].round(4) * 100)

In [ ]:
os.chdir(WORKDIR)
import pandas as pd

tiles_rows = []
for split in ("val", "test"):
    m = json.loads(Path(f"preds/slice/slice_meta_{split}.json").read_text())
    tiles_rows.append({"split": split, "device": m["device"],
                       "tiles/image": m.get("tiles_per_image_mean"),
                       "ms/image full view (hflip TTA)": m["ms_per_image"].get("full"),
                       "ms/image all tiles": m["ms_per_image"].get("tiles")})
print("Sliced inference wall-clock on this device (not the published CPU latency):")
display(pd.DataFrame(tiles_rows).set_index("split"))

casc_rows = []
for p in sorted(Path("preds").glob("cascade_*/cascade_meta_*_*.json")):
    c = json.loads(p.read_text())
    casc_rows.append({"dump": p.parent.name, "mode": c["mode"],
                      "dets scored": c["num_output"], "seconds": c["seconds"]})
if casc_rows:
    print()
    print("Crop refiner wall-clock:")
    display(pd.DataFrame(casc_rows))

pub = Path("viscan_beat_benchmark_results.zip")
members = []
for folder in list(Path("evaluation").rglob("*.json")) + list(Path("preds").rglob("*.json")):
    members.append((folder, str(folder)))
for extra in ("checkpoints/cascade/convnext_cascade.pth", "checkpoints/cascade/crop_bank_meta.json",
              "checkpoints/retinanet/best_model.pth", "runs/retinanet/loss_curve.png",
              "runs/retinanet/val_ap_curve.png"):
    if Path(extra).exists():
        members.append((Path(extra), extra))
for log in Path(".").glob("*.log"):
    members.append((log, log.name))

import zipfile
with zipfile.ZipFile(pub, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for src, arc in members:
        zf.write(src, arc)
print(f"zip: {pub.resolve()} ({pub.stat().st_size / 1024**2:.1f} MB, {len(members)} files)")

md_lines = ["| pipeline | val AP | val AP50 | test AP | test AP50 | test AP75 | test AR@100 | tau | test macro-F1 |",
            "|---|---|---|---|---|---|---|---|---|"]
for name, r in table.iterrows():
    md_lines.append("| " + name + " | " + " | ".join(f"{float(r[c]):.4f}" for c in
                    ["val AP", "val AP50", "test AP", "test AP50", "test AP75", "test AR@100", "tau", "test macro-F1"]) + " |")
md_table = "\n".join(md_lines)
Path("report_rows.md").write_text(md_table, encoding="utf-8")
print("\nMeasured rows for REPORT.md (paste only after reading them):\n")
print(md_table)

## Reading the result honestly

1. **Selection:** the pipeline chosen above comes from validation mAP@50:95 only. If its test numbers are worse
   than P0, that is the measured outcome — write it down as such, do not re-select on test.
2. **Reproduction:** P0 should land within ~0.002-0.003 of the published 0.496 / 0.329. A larger drift means the
   environment changed something (torchvision version, preprocessing, downloaded weights); stop and investigate
   before trusting any other row.
3. **Latency:** sliced inference multiplies forward passes by the tile count and the refiner adds a classifier
   pass per box. Even if AP improves, the deployed single-pass model stays the deployment default unless the
   gain justifies the cost. The numbers above are wall-clock on this GPU, not the published CPU latency.
4. **Comparability:** this test split has 50 images (182 boxes); the DENTEX leaderboard has 250 hidden images.
   Never present our AP50 as a leaderboard position.
5. **Paste policy:** copy the measured table into `REPORT.md`; mark which rows were measured in this session and
   which came from the earlier report. No number may be written before this run produces it.

In [ ]:
os.chdir(WORKDIR)
import shutil

assert Path("viscan_beat_benchmark_results.zip").exists(), "build the zip before cleanup"
for p in ("data/dentex/zips", "data/dentex/extracted_train", "data/dentex/extracted_val"):
    target = Path(p)
    if target.exists():
        shutil.rmtree(target)
        print("removed", target)

print("final /kaggle/working size:",
      round(sum(f.stat().st_size for f in Path("/kaggle/working").rglob("*") if f.is_file()) / 1024**3, 2),
      "GB")